<a href="https://colab.research.google.com/github/njones61/ce544/blob/main/docs/unit1/06_wells/files/transient_KEY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Well Equations - Transient Drawdown

Instructions: Compute the head around a well over time with the Theis equation, for the confined aquifer of Part 2 and the unconfined aquifer of Part 3. Compare each with the steady solution for R = 500 m.

Confined:

$
h = H - \frac{q}{4\pi T}W(u), \qquad u = \frac{r^2 S}{4Tt}, \qquad T = kD
$

Unconfined:

$
h = \sqrt{H^2 - \frac{q}{2\pi k}W(u)}, \qquad u = \frac{r^2 S_y}{4kHt}
$

At late time, both become the steady equations with $R = 1.5\sqrt{Tt/S}$ (unconfined: $R = 1.5\sqrt{kHt/S_y}$).

$W(u)$ is the exponential integral, `scipy.special.exp1`.

In [ ]:
# import libraries needed
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import exp1   # W(u)

## Confined

In [ ]:
# confined aquifer (Part 2), with a storativity
q = 0.2      #[m^3/s]
k = 0.1      #[cm/s]
k = k/100    #[m/s]
D = 15       #[m]
T = k*D      #[m^2/s]
S = 2e-4     #[-]
H = 50       #[m]
rw = 0.1     #[m]
R = 500      #[m] (steady solution)
times = [60, 600, 3600, 86400]   #[s]: 1 min, 10 min, 1 hour, 1 day
r = np.array([0.1, 1.25, 2.5, 5, 10, 25, 50, 75, 100, 150, 200, 300, 400, 500, 600, 700, 800, 900, 1000])

In [ ]:
# Theis head function
def h(r, t):
    u = r**2*S/(4*T*t)
    return H - q/(4*np.pi*T)*exp1(u)

# Thiem (steady), R = 500
def h_thiem(r):
    return np.where(r < R, H + q/(2*np.pi*T)*np.log(r/R), H)

In [ ]:
for t in times:
    print(f't = {t:>6} s: head at the well = {h(rw, t):.2f} m, R = {1.5*np.sqrt(T*t/S):.0f} m')
print(f'Thiem, R = 500 m: head at the well = {h_thiem(rw):.2f} m')

In [ ]:
tR = (R/1.5)**2*S/T
print(f'R reaches {R} m after {tR:.0f} s = {tR/60:.1f} min; head at the well then = {h(rw, tR):.2f} m')

In [ ]:
plt.figure(figsize=(10,4))
for t in times:
    plt.plot(r, h(r, t), label=f't = {t} s')
plt.plot(r, h_thiem(r), 'k--', label='Thiem, R = 500 m')
plt.xlabel('r [m]')
plt.ylabel('Head [m]')
plt.title('Confined: head vs r over time')
plt.legend()
plt.show()

## Unconfined

In [ ]:
# unconfined aquifer (Part 3, with a lower k), with a specific yield
q = 0.2      #[m^3/s]
k = 0.05     #[cm/s]
k = k/100    #[m/s]
H = 50       #[m]
Sy = 0.2     #[-]
rw = 0.1     #[m]
R = 500      #[m] (steady solution)
times = [3600, 86400, 10*86400, 100*86400]   #[s]: 1 hour, 1 day, 10 days, 100 days

In [ ]:
def h(r, t):
    u = r**2*Sy/(4*k*H*t)
    return np.sqrt(H**2 - q/(2*np.pi*k)*exp1(u))

def h_steady(r):
    return np.where(r < R, np.sqrt(H**2 - q/(np.pi*k)*np.log(R/r)), H)

for t in times:
    print(f't = {t:>6} s: head at the well = {h(rw, t):.3f} m, R = {1.5*np.sqrt(k*H*t/Sy):.0f} m')
print(f'Steady, R = 500 m: head at the well = {h_steady(rw):.3f} m')
tR = (R/1.5)**2*Sy/(k*H)
print(f'R reaches {R} m after {tR:.0f} s = {tR/86400:.1f} days')

In [ ]:
plt.figure(figsize=(10,4))
for t in times:
    plt.plot(r, h(r, t), label=f't = {t} s')
plt.plot(r, h_steady(r), 'k--', label='Steady, R = 500 m')
plt.xlabel('r [m]')
plt.ylabel('Head [m]')
plt.title('Unconfined: head vs r over time')
plt.legend()
plt.show()